# Módulo 2 — Corpus preprocesado y EDA técnico

**Objetivo:** dejar el corpus AG News listo para modelado — limpieza Regex,
normalización y lematización con SpaCy — y caracterizarlo: longitud de documentos,
n-gramas dominantes, vocabulario frecuente y balance de clases. Las decisiones que
salen de aquí (vocabulario limpio, percentil 95 de longitud) alimentan los Módulos 3 y 4.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.en.stop_words import STOP_WORDS

from src.config import FIGURES_DIR, PROCESSED_DIR, LABEL_NAMES
from src.dataset import load_raw
from src.preprocessing import preprocess_corpus

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carga y pipeline de preprocesamiento

`preprocess_corpus` (en `src/preprocessing.py`) encadena: lowercase → remoción de
HTML/URLs/caracteres no alfabéticos → lematización SpaCy → filtro de stop-words y
puntuación. El filtro de longitud conserva tokens de 2 letras ("us", "uk", "eu"):
en un corpus de noticias son marcadores fuertes de la clase World y quitarlos
sería exactamente el error de "cola larga" que la consigna advierte — filtrar
vocabulario con carga semántica del dominio.

In [ ]:
train_df, test_df = load_raw()
print(f"train: {len(train_df):,} docs | test: {len(test_df):,} docs")

In [ ]:
# Corpus completo. Con parser y NER deshabilitados el pipe procesa por lotes.
train_df["text_clean"] = preprocess_corpus(train_df["text"], batch_size=512, n_process=1)
test_df["text_clean"] = preprocess_corpus(test_df["text"], batch_size=512, n_process=1)

train_df.to_csv(PROCESSED_DIR / "train_clean.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test_clean.csv", index=False)
print("corpus procesado guardado en data/processed/")

In [ ]:
muestra = train_df[["text", "text_clean"]].sample(3, random_state=42)
for _, row in muestra.iterrows():
    print("ORIGINAL :", row["text"][:120])
    print("LIMPIO   :", row["text_clean"][:120])
    print("-" * 80)

## 2. Longitud de documentos y percentil 95

El p95 sobre el texto limpio describe el corpus tras el preprocesamiento. Para el
`max_len` del Módulo 4 este número **no se usa tal cual**: el Transformer tokeniza
el texto crudo en subword tokens (las stop-words vuelven y las palabras se parten),
así que allá el p95 se recalcula con el propio tokenizador de Hugging Face.

In [ ]:
lengths = train_df["text_clean"].str.split().apply(len)
p95 = int(np.percentile(lengths, 95))
print(f"Percentil 95 de tokens (texto limpio): {p95}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lengths, bins=60)
ax.axvline(p95, linestyle="--", color="crimson", label=f"p95 = {p95}")
ax.set_xlabel("tokens por documento (texto limpio)")
ax.set_ylabel("frecuencia")
ax.set_title("Módulo 2 — distribución de longitud de documentos")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "m2_hist_tokens.png", dpi=150)
plt.show()

## 3. N-gramas más frecuentes

In [ ]:
def top_ngrams(corpus, n, k=20):
    vec = CountVectorizer(ngram_range=(n, n)).fit(corpus)
    bow = vec.transform(corpus)
    sums = bow.sum(axis=0).A1
    pares = sorted(zip(vec.get_feature_names_out(), sums), key=lambda x: -x[1])
    return pares[:k]


top_bi = top_ngrams(train_df["text_clean"], 2, 20)
top_tri = top_ngrams(train_df["text_clean"], 3, 20)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, data, title in [(axes[0], top_bi, "Top 20 bi-gramas"),
                        (axes[1], top_tri, "Top 20 tri-gramas")]:
    names = [t for t, _ in data][::-1]
    counts = [c for _, c in data][::-1]
    ax.barh(names, counts)
    ax.set_title(title)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "m2_ngrams.png", dpi=150)
plt.show()

## 4. Top 50 palabras y control de stop-words

Si la limpieza funcionó, ninguna de las 50 palabras más frecuentes debería ser una
stop-word estándar — este conteo es el control de calidad del pipeline.

In [ ]:
tokens = " ".join(train_df["text_clean"]).split()
top50 = Counter(tokens).most_common(50)
n_stop = sum(1 for w, _ in top50 if w in STOP_WORDS)
print(f"Stop-words dentro del top 50: {n_stop}")
print([w for w, _ in top50])

## 5. Distribución de clases

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x="label_name", data=train_df, order=LABEL_NAMES, ax=ax)
ax.set_xlabel("categoría")
ax.set_ylabel("documentos")
ax.set_title("Módulo 2 — distribución de documentos por categoría (train)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "m2_class_dist.png", dpi=150)
plt.show()

print(train_df["label_name"].value_counts())

## 6. Conclusión

<!-- CONCLUSION_M2: completar tras la corrida (máx. 300 palabras) -->